# BREAST CANCER CLASSIFICATION: 
### LOGISTIC REGRESSION VS NEURAL NETWORK (PYTORCH)

### Objetive: 
Create a multilayered neural network to predict ... breast cancer, at this point it's clear that I like dealing with binary classification datasets, I have enough headaches dealing with college.
And then comparing a Logistic Regression model to see which model gets the best predictions and consider which is the best model considering their tade-offs

My name: Emmanuel Saucedo

Date: 20 Feb 2026


#### This notebook solves:
A breast cancer prediction model using multiple activation functions such as:
 * Linear Functions
 * ReLU (Rectified Linear Unit)
 * Final Sigmoid Function for binary predictions

And another model using Logistic Regression solving the same problem




## Creating the Neural Network

In [1]:
import torch 
from torch import nn
import numpy as np

import random

def set_seed(seed=777):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

class Model(nn.Module):
    def __init__(self, n_inputs : int = 4):
        super(Model, self).__init__()
        #Note I decided h1 for hidden no.K
        self.h1 = nn.Linear(n_inputs, 5)
        self.h2 = nn.ReLU() #Note for me: This is the first time I use ReLU AF how amazing (It's not amazing, literally is boring AF[Activation Function])
        self.h3 = nn.Linear(5,1)
        self.h4 = nn.Sigmoid()
    def forward(self, x):
        x = self.h1(x)
        x = self.h2(x)
        x = self.h3(x)
        x = self.h4(x)
        return x

#Probably I'll fail too many times, I'll record how many times I fail with my code
# IIIII_IIIII_II

In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X, Y = load_breast_cancer(return_X_y= True)

X = StandardScaler().fit_transform(X)


X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size= .8, random_state= 42)

def ToTorch(X, Y):
    X = torch.from_numpy(X).float()
    Y = torch.from_numpy(Y).float().unsqueeze(1)
    return X, Y

#Torchinginginging

X_train_PT, Y_train_PT = ToTorch(X_train, Y_train)
X_test_PT, Y_test_PT = ToTorch(X_test, Y_test)

model = Model(X_train_PT.shape[1])
optimizer = torch.optim.SGD(
    model.parameters(),
    lr = 0.01, )
criteria = nn.BCELoss()

epochs = 3_001

for epoch in range(epochs):
    optimizer.zero_grad()
    A = model(X_train_PT)
    loss = criteria(A, Y_train_PT)
    loss.backward()
    optimizer.step()
    if not epoch % 500:
        print(f"I the epoch ({epoch}) the los was: {loss:.3f}")


I the epoch (0) the los was: 0.667
I the epoch (500) the los was: 0.191
I the epoch (1000) the los was: 0.108
I the epoch (1500) the los was: 0.083
I the epoch (2000) the los was: 0.072
I the epoch (2500) the los was: 0.065
I the epoch (3000) the los was: 0.060


## Creating the Logistic Regression model

In [3]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

LR_grid = {
    "l1_ratio" : [1,0], 
    "max_iter" : [100,1000, 5000],
    "C" : [.05, .1 , 1],
    "solver" : ["liblinear"] 
}


LR_clf = GridSearchCV(
    LogisticRegression(random_state= 777),
    LR_grid,
    cv = 5,
    scoring = {
        "recall": "recall",
        "roc_auc": "roc_auc"
    },
    refit = "recall"
)

LR_clf.fit(X_train, Y_train)

LR_model = LR_clf.best_estimator_

In [4]:
import pandas as pd
from sklearn.metrics import accuracy_score

Y_pd = pd.DataFrame(Y)

print(f"""
Comparation of positives and negatives in the dataset:

{Y_pd.value_counts()}
The dataset is not very far from being balanced
      """)

model.eval()
with torch.no_grad():
    pred = model(X_test_PT)
    preds = (pred > 0.5).float()
    accuracy = (preds == Y_test_PT).float().mean()

print(f"""
Comparing the models:
    
    
    Neural Network:
        Accuracy: {accuracy.item() * 100:.6f}%

    Logistic Regression Model:
        Accuracy: {accuracy_score(Y_test, LR_model.predict(X_test)) * 100:.6f}%
""")



Comparation of positives and negatives in the dataset:

0
1    357
0    212
Name: count, dtype: int64
The dataset is not very far from being balanced
      

Comparing the models:


    Neural Network:
        Accuracy: 99.122804%

    Logistic Regression Model:
        Accuracy: 99.122807%



## Is Logistic Regression the future of AI?

Well the answer is surely NO, yes the simple and less complex model achieved a higher accuracy while the complex NN had a 0.000003% less accuracy, but this shows that the data is very linearly separable and opting for a simpliest model is the best option, considering that both got the same results.

And probably the difference between the two models is just a little offset of the python decimals.  

# Conclusion
While both model did great on the predictions the best model for this situation is the __Logistic Regression__ model because its simplicity.